# ⚠️ Notebook histórico — superado

Este notebook conserva evidencia de la primera comprobación. **No es el entregable final.**

La reserva Dataset B v1 usaba el split `train` de FLEURS y fue reemplazada por el split oficial `test`:

`data/processed/manifests/fleurs_evaluation_v2_official_test.csv`

Use `notebooks/Proyecto_IA_Traductor_MASTER_COLAB.ipynb` para la ejecución final.


# Comprobación formal — Dataset A: OPUS-100 NMT

Ejecute desde una sesión nueva de Google Colab. No reutiliza smoke tests ni ejecuta entrenamiento full.

# 1. Inicialización

In [1]:
import os,sys,subprocess,platform,json,time,random,shutil
from pathlib import Path
URL="https://github.com/Setrox1250/Proyecto-IA-Traductor.git"; ROOT=Path("/content/Proyecto-IA-Traductor")
if not ROOT.exists(): subprocess.run(["git","clone",URL,str(ROOT)],check=True)
else:
    if not (ROOT/".git").exists(): raise RuntimeError("Existe /content/Proyecto-IA-Traductor pero no es Git.")
    subprocess.run(["git","-C",str(ROOT),"fetch","--all","--prune"],check=True)
    br=subprocess.check_output(["git","-C",str(ROOT),"branch","--show-current"],text=True).strip()
    if not br: raise RuntimeError("Clon detached; no se cambia rama automáticamente.")
    dirty=subprocess.check_output(["git","-C",str(ROOT),"status","--porcelain"],text=True).strip()
    if dirty: print("El clon contiene trabajo local; se conserva HEAD y no se ejecuta pull.")
    else: subprocess.run(["git","-C",str(ROOT),"pull","--ff-only","origin",br],check=True)
os.chdir(ROOT); branch=subprocess.check_output(["git","branch","--show-current"],text=True).strip(); commit=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip(); log1=subprocess.check_output(["git","log","-1","--oneline"],text=True).strip()
print(branch,commit,log1,sep="\n")

main
c8041497cd7d194eb5d1417f7f8d8b5f2cda273c
c804149 entrega 2


# 2. Clonado del repositorio

In [2]:
# El clonado seguro y la traza Git se ejecutaron en la sección 1.

# 3. Entorno

In [3]:
subprocess.run([sys.executable,"-m","pip","install","-q","-U","datasets>=3","pyarrow<19","evaluate","sacrebleu","jiwer","sentencepiece","sacremoses","accelerate","soundfile","matplotlib","transformers==4.57.3","coqui-tts==0.27.5","speechbrain==1.1.1"],check=True)
import importlib.metadata as im,torch,torchaudio,transformers,datasets,sacrebleu,jiwer,numpy as np
OUT=Path("outputs/comprobacion/datasetA"); FIG=OUT/"figures"; FIG.mkdir(parents=True,exist_ok=True)
def package_version(name):
 try:
  return im.version(name)
 except im.PackageNotFoundError:
  return None
env={"timestamp_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),"python":sys.version,"operating_system":platform.platform(),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"vram_total_gib":round(torch.cuda.get_device_properties(0).total_memory/2**30,3) if torch.cuda.is_available() else None,"torch":torch.__version__,"cuda":torch.version.cuda,"torchaudio":torchaudio.__version__,"transformers":transformers.__version__,"datasets":datasets.__version__,"sacrebleu":package_version("sacrebleu"),"jiwer":package_version("jiwer"),"coqui_tts":package_version("coqui-tts"),"speechbrain":package_version("speechbrain"),"numpy":np.__version__,"git_branch":branch,"git_commit":commit,"git_log_1":log1}
(OUT/"environment.json").write_text(json.dumps(env,indent=2),encoding="utf8")
if not torch.cuda.is_available(): raise RuntimeError("Se requiere GPU Colab.")
DEV=torch.device("cuda")

RuntimeError: Se requiere GPU Colab.

# 4. Dataset A

In [ ]:
import pandas as pd
from src.dataset.parallel import load_opus100,prepare_opus100_splits
S=prepare_opus100_splits(load_opus100(streaming=True),train_size=100000,seed=42); exp={"train":100000,"validation":1990,"test":1999}; got={k:len(v) for k,v in S.items()}
if got!=exp: raise RuntimeError(f"OPUS-100 discrepante: {got}, esperado {exp}")
audit={}; names=list(S)
for k,d in S.items():
    if list(d.columns)!=["text_es","text_en"]: raise RuntimeError(f"Columnas de {k}: {d.columns}")
    pairs=list(zip(d.text_es,d.text_en)); audit[k]={"empty":int(((d.text_es.str.strip()=="")|(d.text_en.str.strip()=="")).sum()),"duplicate_pairs":len(pairs)-len(set(pairs))}
    if any(audit[k].values()): raise RuntimeError(f"Datos inválidos {k}: {audit[k]}")
for i,a in enumerate(names):
    for b in names[i+1:]:
        n=len(set(S[a].text_es)&set(S[b].text_es))+len(set(S[a].text_en)&set(S[b].text_en))
        if n: raise RuntimeError(f"Solapamiento {a}/{b}: {n}")
summary={"dataset":"Helsinki-NLP/opus-100","config":"en-es","direction":"es->en","seed":42,"counts":got,"audit":audit,"git_commit":commit}; (OUT/"dataset_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf8")

# 5. Baseline

In [ ]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
M="Helsinki-NLP/opus-mt-es-en"; TOK=AutoTokenizer.from_pretrained(M); base=AutoModelForSeq2SeqLM.from_pretrained(M).to(DEV).eval()
def evaluate(model,df):
 p=[]; t=time.perf_counter(); tokens=0; loss_sum=0; nlab=0
 for i in range(0,len(df),16):
  x=df.text_es.iloc[i:i+16].tolist(); y=df.text_en.iloc[i:i+16].tolist(); z=TOK(x,return_tensors="pt",padding=True,truncation=True,max_length=128); z={k:v.to(DEV) for k,v in z.items()}
  with torch.inference_mode():
   g=model.generate(**z,max_length=128,num_beams=4); lab=TOK(text_target=y,return_tensors="pt",padding=True,truncation=True,max_length=128).input_ids.to(DEV); lab[lab==TOK.pad_token_id]=-100; l=model(**z,labels=lab).loss
  q=[s.strip() for s in TOK.batch_decode(g,skip_special_tokens=True)]; p+=q; tokens+=sum(map(len,TOK(q,add_special_tokens=False).input_ids)); loss_sum+=l.item()*(lab!=-100).sum().item(); nlab+=(lab!=-100).sum().item()
 sec=time.perf_counter()-t; return p,{"num_samples":len(df),"sacrebleu":sacrebleu.corpus_bleu(p,[df.text_en.tolist()]).score,"chrf":sacrebleu.corpus_chrf(p,[df.text_en.tolist()]).score,"eval_loss":loss_sum/nlab,"total_seconds":sec,"mean_seconds_per_sentence":sec/len(df),"output_tokens_per_second":tokens/sec,"num_beams":4,"max_length":128,"git_commit":commit}
test=S["test"].reset_index(drop=True); pb,mb=evaluate(base,test); pd.DataFrame({"sample_id":range(1999),"source_es":test.text_es,"reference_en":test.text_en,"prediction_base":pb}).to_csv(OUT/"predictions_test_base.csv",index=False); (OUT/"baseline_metrics.json").write_text(json.dumps(mb,indent=2),encoding="utf8")

# 6. Controlled training

In [ ]:
from datasets import Dataset
import gc
from transformers import DataCollatorForSeq2Seq,Seq2SeqTrainer,Seq2SeqTrainingArguments,set_seed
set_seed(42); attempts=[]
for n in (10000,5000,2500):
 try:
  model=AutoModelForSeq2SeqLM.from_pretrained(M).to(DEV); tok=lambda d: Dataset.from_pandas(d[["text_es","text_en"]],preserve_index=False).map(lambda x:dict(TOK(x["text_es"],truncation=True,max_length=128),labels=TOK(text_target=x["text_en"],truncation=True,max_length=128)["input_ids"]),batched=True,remove_columns=["text_es","text_en"])
  args=Seq2SeqTrainingArguments(output_dir=str(OUT/"controlled_checkpoint"),learning_rate=5e-5,weight_decay=.01,label_smoothing_factor=.1,per_device_train_batch_size=8,per_device_eval_batch_size=16,num_train_epochs=1,fp16=True,eval_strategy="steps",eval_steps=200,logging_steps=50,save_strategy="no",seed=42,data_seed=42,report_to="none",optim="adamw_torch")
  tr=Seq2SeqTrainer(model=model,args=args,train_dataset=tok(S["train"].iloc[:n]),eval_dataset=tok(S["validation"].iloc[:1000]),data_collator=DataCollatorForSeq2Seq(TOK,model=model,pad_to_multiple_of=8,label_pad_token_id=-100),processing_class=TOK)
  torch.cuda.reset_peak_memory_stats(); st=time.perf_counter(); r=tr.train(); train_sec=time.perf_counter()-st; used=n; break
 except RuntimeError as e:
  attempts.append({"train_attempt":n,"status":"cuda_oom_fallback" if "out of memory" in str(e).lower() else "error","error":str(e)}); tr=None; model=None; gc.collect(); torch.cuda.empty_cache()
  if "out of memory" not in str(e).lower() or n==2500: raise
cfg={"experiment_name":"MarianMT — Controlled Training Checkpoint","not_a_final_model":True,"train_available":100000,"train_used":used,"validation_used":1000,"epochs":1,"learning_rate":5e-5,"weight_decay":.01,"label_smoothing":.1,"fp16":True,"optimizer":"AdamW","dynamic_padding":True,"num_beams":4,"max_length":128,"seed":42,"fallback_attempts":attempts,"training_seconds":train_sec,"training_loss":r.training_loss,"peak_vram_gib":torch.cuda.max_memory_allocated()/2**30,"git_commit":commit}; (OUT/"training_config.json").write_text(json.dumps(cfg,indent=2),encoding="utf8"); model.save_pretrained(OUT/"controlled_checkpoint"); TOK.save_pretrained(OUT/"controlled_checkpoint")

# 7. Test completo

In [ ]:
model.eval(); pc,mc=evaluate(model,test); mc.update({"checkpoint":"MarianMT — Controlled Training Checkpoint","train_used":used}); trained_frame=pd.DataFrame({"sample_id":range(1999),"source_es":test.text_es,"reference_en":test.text_en,"prediction_controlled":pc}); trained_frame.to_csv(OUT/"predictions_test_trained.csv",index=False); pd.DataFrame({"sample_id":range(1999),"source_es":test.text_es,"reference_en":test.text_en,"prediction_base":pb,"prediction_controlled":pc}).to_csv(OUT/"predictions_test.csv",index=False); (OUT/"trained_metrics.json").write_text(json.dumps(mc,indent=2),encoding="utf8")

# 8. Comparación

In [ ]:
comp={"same_test_samples":1999,"base":mb,"controlled":mc,"delta":{k:mc[k]-mb[k] for k in ["sacrebleu","chrf","eval_loss","total_seconds","mean_seconds_per_sentence"]}}; (OUT/"comparison.json").write_text(json.dumps(comp,indent=2),encoding="utf8"); pd.DataFrame([{"Métrica":k,"Base":mb[k],"Controlled":mc[k],"Delta":comp["delta"][k]} for k in ["sacrebleu","chrf","eval_loss","total_seconds"]])

# 9. Gráficos

In [ ]:
import matplotlib.pyplot as plt
h=tr.state.log_history; x=[(z["step"],z["loss"]) for z in h if "loss" in z]; plt.plot(*zip(*x));plt.xlabel("Paso");plt.ylabel("Training loss");plt.tight_layout();plt.savefig(FIG/"training_loss.png",dpi=160);plt.close()
v=[(z["step"],z["eval_loss"]) for z in h if "eval_loss" in z]
if len(v)>=2: plt.plot(*zip(*v));plt.xlabel("Paso");plt.ylabel("Validation loss");plt.tight_layout();plt.savefig(FIG/"validation_loss.png",dpi=160);plt.close()
plt.bar(["Base BLEU","Controlled BLEU","Base chrF","Controlled chrF"],[mb["sacrebleu"],mc["sacrebleu"],mb["chrf"],mc["chrf"]]);plt.xticks(rotation=15);plt.tight_layout();plt.savefig(FIG/"base_vs_controlled_metrics.png",dpi=160);plt.close()

# 10. Exportación

In [ ]:
print("Dataset A terminado:",OUT)
print("Checkpoint excluido del ZIP final de Dataset B.")